# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)

(1338, 7)
['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']
age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object


## A.2. Missing values & Duplicate data

In [3]:
print(df.isna().sum())
print('Duplicates:', df.duplicated().sum())

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64
Duplicates: 1


## A.3. Invalid values

In [4]:
print('Age < 0:', (df.age < 0).sum())
print('BMI <= 0:', (df.bmi <= 0).sum())
print('Children < 0:', (df.children < 0).sum())
print('Charges <= 0:', (df.charges <= 0).sum())
print('Sex:', df.sex.unique())
print('Smoker:', df.smoker.unique())

Age < 0: 0
BMI <= 0: 0
Children < 0: 0
Charges <= 0: 0
Sex: <StringArray>
['female', 'male']
Length: 2, dtype: str
Smoker: <StringArray>
['yes', 'no']
Length: 2, dtype: str


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [5]:
df['bmi_group'] = pd.cut(df['bmi'], [-np.inf, 25, 30, np.inf], labels=['Normal', 'Overweight', 'Obese'])
df[['bmi', 'bmi_group']].head()

,bmi,bmi_group
0,27.900,Overweight
1,33.770,Obese
2,33.000,Obese
3,22.705,Normal
4,28.880,Overweight


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [6]:
print(df[['age', 'bmi', 'children', 'charges']].mean())
print(df[['age', 'bmi', 'children', 'charges']].median())
print(df[['age', 'bmi', 'children', 'charges']].mode().iloc[0])

age            39.207025
bmi            30.663397
children        1.094918
charges     13270.422265
dtype: float64
age           39.000
bmi           30.400
children       1.000
charges     9382.033
dtype: float64
age           18.0000
bmi           32.3000
children       0.0000
charges     1639.5631
Name: 0, dtype: float64


## Group 2 — Dispersion

In [7]:
print(df[['age', 'bmi', 'children', 'charges']].std())
print(df[['age', 'bmi', 'children', 'charges']].var())
print(df[['age', 'bmi', 'children', 'charges']].min())
print(df[['age', 'bmi', 'children', 'charges']].max())

age            14.049960
bmi             6.098187
children        1.205493
charges     12110.011237
dtype: float64
age         1.974014e+02
bmi         3.718788e+01
children    1.453213e+00
charges     1.466524e+08
dtype: float64
age           18.0000
bmi           15.9600
children       0.0000
charges     1121.8739
dtype: float64
age            64.00000
bmi            53.13000
children        5.00000
charges     63770.42801
dtype: float64


## Group 3 — Location and Shape

In [8]:
print(df[['age', 'bmi', 'children', 'charges']].quantile([0.25, 0.5, 0.75]))
print(df[['age', 'bmi', 'children', 'charges']].skew())
print(df[['age', 'bmi', 'children', 'charges']].kurtosis())

       age       bmi  children       charges
0.25  27.0  26.29625       0.0   4740.287150
0.50  39.0  30.40000       1.0   9382.033000
0.75  51.0  34.69375       2.0  16639.912515
age         0.055673
bmi         0.284047
children    0.938380
charges     1.515880
dtype: float64
age        -1.245088
bmi        -0.050732
children    0.202454
charges     1.606299
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [9]:
result = df.groupby('smoker')['charges'].mean()
print(result)
print('So lan:', result['yes'] / result['no'])
print(df.groupby(['region', 'smoker'])['charges'].mean().unstack())

smoker
no      8434.268298
yes    32050.231832
Name: charges, dtype: float64
So lan: 3.800001458298319
smoker              no           yes
region                              
northeast  9165.531672  29673.536473
northwest  8556.463715  30192.003182
southeast  8032.216309  34844.996824
southwest  8019.284513  32269.063494


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [10]:
print(df.groupby('smoker')[['bmi', 'charges']].corr().loc[(slice(None), 'bmi'), 'charges'])

smoker     
no      bmi    0.084037
yes     bmi    0.806481
Name: charges, dtype: float64


## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [11]:
print(df.groupby('region')['charges'].mean().sort_values(ascending=False))

region
southeast    14735.411438
northeast    13406.384516
northwest    12417.575374
southwest    12346.937377
Name: charges, dtype: float64


## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [12]:
print(df.groupby('children')['charges'].agg(['mean', 'count']))
print(df[['children', 'charges']].corr().iloc[0, 1])

                  mean  count
children                     
0         12365.975602    574
1         12731.171832    324
2         15073.563734    240
3         15355.318367    157
4         13850.656311     25
5          8786.035247     18
0.06799822684790495


## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [13]:
print(df[['age', 'charges']].corr().iloc[0, 1])
print(df.groupby(pd.cut(df.age, [0, 30, 50, 100]))['charges'].mean())

0.299008193330648
age
(0, 30]       9397.552051
(30, 50]     13280.774031
(50, 100]    18084.987223
Name: charges, dtype: float64


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

Dữ liệu có 1.338 bản ghi và 7 biến, không phát hiện giá trị thiếu hay giá trị âm ở các biến số chính. Chi phí bảo hiểm của người hút thuốc cao hơn rõ rệt so với người không hút thuốc. BMI và tuổi đều có xu hướng cùng chiều với chi phí, trong đó ảnh hưởng của hút thuốc nổi bật hơn. Chi phí trung bình khác nhau giữa các vùng và thay đổi theo số con, nhưng mức độ chênh lệch không lớn bằng yếu tố hút thuốc.